# Gemma4-Legal TensorRT INT4 Optimization

**Goal**: Convert `gemma4-legal:latest` (11.8B Q4_K_M) to TensorRT INT4 format for 3-5× speedup

**Current Performance** (RTX 3060 Ti):
- Ollama: 34.3s per request (56.7 tokens/sec)
- Target: <10s per request (>150 tokens/sec)

**Expected Outcome**:
- TensorRT INT4: 5-10s per request (3-5× faster)
- Enables load testing with production model
- Reduces from 68 GPUs → 10-14 GPUs for 12,000 QPM

---

## Prerequisites

**Hardware**:
- RTX 3060 Ti (8GB VRAM) — Local development
- OR Colab T4/A100/L4 — Cloud conversion

**Software**:
```bash
pip install tensorrt-llm==0.12.0
pip install transformers accelerate bitsandbytes
pip install onnx onnxruntime-gpu
```

**Model Files**:
- Local: `~/.ollama/models/blobs/sha256-*` (GGUF format)
- Will convert: GGUF → Hugging Face → TensorRT

## Step 1: Environment Setup

In [ ]:
# Check GPU availability
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"CUDA Version: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Install TensorRT-LLM (requires CUDA 12.x)
!pip install --quiet tensorrt-llm==0.12.0
!pip install --quiet transformers accelerate bitsandbytes safetensors
!pip install --quiet onnx onnxruntime-gpu

## Step 2: Export Ollama GGUF → Hugging Face Format

**Strategy**: Use llama.cpp's `convert-hf-to-gguf.py` in reverse

**Alternative**: Download base Gemma 4 (9.2B) from Hugging Face and apply LoRA adapter

In [ ]:
# Option A: Download base Gemma 4 9.2B model (recommended for clean conversion)
from huggingface_hub import snapshot_download
import os

model_id = "google/gemma-2-9b-it"  # Base Gemma 2 9B Instruct
model_path = "./models/gemma4-base"

print(f"Downloading {model_id}...")
snapshot_download(
    repo_id=model_id,
    local_dir=model_path,
    local_dir_use_symlinks=False,
    ignore_patterns=["*.gguf", "*.bin"],  # Skip GGUF files, download safetensors only
)
print(f"✅ Model downloaded to {model_path}")

# Note: If using fine-tuned gemma4-legal, you'll need to apply LoRA adapter here
# See: https://huggingface.co/docs/peft/main/en/package_reference/lora

In [ ]:
# Option B: Extract GGUF from Ollama and convert (advanced)
# This requires llama.cpp's convert scripts and is more complex
# Recommended: Use Option A + apply fine-tune adapter instead

# Find Ollama model blob
!ls ~/.ollama/models/blobs/ | grep sha256 | head -5

# Manual steps (not automated):
# 1. Identify correct blob (largest file, ~7GB for gemma4-legal)
# 2. Copy to workspace: cp ~/.ollama/models/blobs/sha256-XXX ./gemma4-legal.gguf
# 3. Clone llama.cpp: git clone https://github.com/ggerganov/llama.cpp
# 4. Run reverse conversion (experimental, may not work)

print("⚠️ Option B requires manual steps. Using Option A (base model) is recommended.")

## Step 3: Apply Legal Fine-Tune (If Available)

**If you have**:
- LoRA adapter weights from GRPO training
- Fine-tuned safetensors checkpoint

**Then**: Merge adapter into base model before TensorRT conversion

In [ ]:
# Load base model + fine-tune adapter (if available)
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Option 1: Base model only (no fine-tune)
base_model_path = "./models/gemma4-base"
model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(base_model_path)

print("✅ Base model loaded")

# Option 2: Merge LoRA adapter (if you have fine-tuned weights)
# adapter_path = "./gemma4-legal-adapter"  # Path to LoRA checkpoint
# model = PeftModel.from_pretrained(model, adapter_path)
# model = model.merge_and_unload()  # Merge adapter into base weights
# print("✅ Fine-tune adapter merged")

## Step 4: INT4 Quantization (AWQ)

**Method**: Activation-aware Weight Quantization (AWQ)

**Why AWQ over GPTQ**:
- Better TensorRT compatibility
- 1-2% accuracy loss vs FP16 (acceptable for legal Q&A)
- 4× memory reduction (11.8B → ~3GB)

In [ ]:
# Install AutoAWQ for INT4 quantization
!pip install --quiet autoawq

from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

# Quantization config
quant_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM"
}

# Load model for quantization
print("Loading model for quantization...")
awq_model = AutoAWQForCausalLM.from_pretrained(
    model_path,
    device_map="cuda:0",
    safetensors=True,
)

# Calibration dataset (legal text samples)
# Use real legal text for better quantization
calibration_data = [
    "What is the definition of hearsay evidence under Federal Rules of Evidence 801?",
    "Explain the attorney-client privilege and its exceptions in civil litigation.",
    "What are the elements required to prove breach of contract in California?",
    "Define negligence per se and provide examples from tort law.",
    "What is the statute of limitations for personal injury claims in New York?",
    # Add more legal queries for better calibration
]

print("Quantizing model to INT4...")
awq_model.quantize(
    tokenizer,
    quant_config=quant_config,
    calib_data=calibration_data,
)

# Save quantized model
quant_path = "./models/gemma4-legal-awq-int4"
awq_model.save_quantized(quant_path)
tokenizer.save_pretrained(quant_path)

print(f"✅ INT4 quantized model saved to {quant_path}")

## Step 5: Export to TensorRT Engine

**TensorRT Engine Features**:
- Fused kernels (multi-head attention, LayerNorm, GELU)
- KV cache optimization
- FP16 activations + INT4 weights
- Inflight batching support

In [ ]:
# Build TensorRT engine from quantized model
import tensorrt_llm
from tensorrt_llm.builder import Builder
from tensorrt_llm.network import net_guard
from tensorrt_llm.plugin.plugin import ContextFMHAType

# TensorRT-LLM config
trt_config = {
    'model_dir': quant_path,
    'dtype': 'float16',  # Activations in FP16
    'quant_mode': 'int4_awq',  # Weights in INT4
    'max_batch_size': 8,
    'max_input_len': 2048,
    'max_output_len': 512,
    'max_beam_width': 1,
    'use_gpt_attention_plugin': 'float16',
    'use_gemm_plugin': 'float16',
    'enable_context_fmha': True,  # Flash Attention
    'context_fmha_type': ContextFMHAType.enabled,
}

# Build engine (this takes 5-15 minutes)
print("Building TensorRT engine... (this may take 10-15 minutes)")
!trtllm-build \
    --checkpoint_dir {quant_path} \
    --output_dir ./engines/gemma4-legal-trt \
    --gemm_plugin float16 \
    --max_batch_size 8 \
    --max_input_len 2048 \
    --max_output_len 512 \
    --use_fused_mlp \
    --enable_context_fmha

print("✅ TensorRT engine built successfully")
print("Engine location: ./engines/gemma4-legal-trt/")

## Step 6: Benchmark Performance

**Compare**:
1. Ollama (baseline): 34.3s per request
2. TensorRT INT4: Target <10s per request

In [ ]:
# Load TensorRT engine for inference
from tensorrt_llm.runtime import ModelRunner
import time

# Initialize runner
engine_dir = "./engines/gemma4-legal-trt"
runner = ModelRunner.from_dir(
    engine_dir=engine_dir,
    rank=0,
    debug_mode=False,
)

# Test queries (legal domain)
test_queries = [
    "What is hearsay evidence in California?",
    "Define attorney-client privilege",
    "What are the elements of breach of contract?",
    "Explain negligence per se",
    "What is the statute of limitations for personal injury?",
]

# Benchmark
latencies = []
print("\n=== TensorRT Inference Benchmark ===")

for i, query in enumerate(test_queries, 1):
    # Tokenize
    input_ids = tokenizer.encode(query, return_tensors="pt").to("cuda")
    
    # Inference
    start = time.time()
    outputs = runner.generate(
        input_ids,
        max_new_tokens=200,
        temperature=0.3,
        top_p=0.9,
    )
    latency = time.time() - start
    latencies.append(latency)
    
    # Decode
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print(f"Query {i}: {latency:.2f}s")
    print(f"Response: {response[:100]}...\n")

# Statistics
avg_latency = sum(latencies) / len(latencies)
print(f"\n=== Results ===")
print(f"Average Latency: {avg_latency:.2f}s")
print(f"Min: {min(latencies):.2f}s")
print(f"Max: {max(latencies):.2f}s")
print(f"\nComparison:")
print(f"  Ollama (baseline): 34.3s")
print(f"  TensorRT INT4:     {avg_latency:.2f}s")
print(f"  Speedup:           {34.3 / avg_latency:.1f}×")

## Step 7: Deploy TensorRT Engine

**Deployment Options**:

### Option A: TensorRT-LLM Triton Server (Recommended)
```bash
# Start Triton server with TensorRT backend
docker run --gpus all --rm -p 8000:8000 -p 8001:8001 -p 8002:8002 \
  -v $(pwd)/engines:/models \
  nvcr.io/nvidia/tritonserver:24.01-trtllm-python-py3 \
  tritonserver --model-repository=/models
```

### Option B: Standalone Python Server
```python
# Create FastAPI server
from fastapi import FastAPI
from tensorrt_llm.runtime import ModelRunner

app = FastAPI()
runner = ModelRunner.from_dir("./engines/gemma4-legal-trt")

@app.post("/generate")
async def generate(prompt: str):
    outputs = runner.generate(prompt, max_new_tokens=200)
    return {"response": outputs}

# Run: uvicorn server:app --host 0.0.0.0 --port 8099
```

### Option C: Update Inference Router
Update `inference-router.ts` to prioritize TensorRT endpoint:
```typescript
// Priority order:
1. TensorRT-LLM (:8099) — Now active!
2. Bifrost Cache (:3040)
3. Ollama (:11434) — Fallback only
```

## Step 8: Integration Test

**Test full stack**:
1. Start TensorRT server (:8099)
2. Update load test to use main `/api/ai/chat` endpoint
3. Verify router prioritizes TensorRT
4. Measure cache hit rates (L1 + L2 + L3)

In [ ]:
# Test TensorRT endpoint via HTTP
import requests
import json

# Assuming TensorRT server is running on :8099
trt_endpoint = "http://localhost:8099/generate"

# Test request
test_payload = {
    "prompt": "What is hearsay evidence in California?",
    "max_tokens": 200,
    "temperature": 0.3,
}

response = requests.post(trt_endpoint, json=test_payload)
if response.status_code == 200:
    result = response.json()
    print(f"✅ TensorRT server responding")
    print(f"Response: {result['response'][:200]}...")
    print(f"Latency: {result.get('latency_ms', 'N/A')}ms")
else:
    print(f"❌ Error: {response.status_code}")
    print(response.text)

## Expected Results

### Performance Targets

| Metric | Ollama (Q4_K_M) | TensorRT INT4 | Improvement |
|--------|-----------------|---------------|-------------|
| **Latency** | 34.3s | 5-10s | 3-5× faster |
| **Throughput** | 1.75 req/min | 6-12 req/min | 3-7× higher |
| **VRAM Usage** | 7.5GB | 3-4GB | 50% reduction |
| **QPM (single GPU)** | ~7 | ~360-720 | 50-100× higher |

### Load Testing Impact

**Before** (Ollama gemma4-legal):
- 34.3s per request
- Need 68 GPUs for 12,000 QPM
- Load tests timeout

**After** (TensorRT INT4):
- 5-10s per request
- Need 10-14 GPUs for 12,000 QPM
- Load tests pass with 100% success rate
- Can validate L1+L2+L3 cache system

### Quality Validation

**Accuracy Loss** (INT4 vs FP16):
- Legal Q&A: 1-2% drop (acceptable)
- Citation extraction: <1% drop
- Reasoning tasks: 2-3% drop

**Recommended**: Run legal benchmark suite after conversion to validate quality

## Troubleshooting

### Error: CUDA Out of Memory
```
Solution: Reduce batch size or max_input_len
trtllm-build --max_batch_size 4 --max_input_len 1024
```

### Error: TensorRT build fails
```
Check: CUDA version compatibility (need 12.1+)
nvidia-smi
```

### Performance not improved
```
Verify:
1. GPU utilization: nvidia-smi dmon
2. Flash Attention enabled: check engine config
3. INT4 quantization active: inspect engine file
```

### Quality degradation >5%
```
Try:
1. Use calibration data with more legal samples
2. Increase q_group_size (128 → 256)
3. Consider INT8 instead of INT4
```

## Next Steps

After successful conversion:

1. **Deploy TensorRT Server** (:8099)
   ```bash
   # Start server
   python trt_server.py
   ```

2. **Update Inference Router**
   ```typescript
   // inference-router.ts: Prioritize TensorRT
   const backends = [
     tryTensorRT,    // NEW: Try first (5-10s)
     tryBifrost,     // L2 cache check
     tryOllama,      // Fallback only
   ];
   ```

3. **Run Load Tests**
   ```bash
   # Full suite with production model
   node scripts/tests/redis-load-test.mjs --duration=300 --concurrency=50
   ```

4. **Validate Cache Layers**
   - Measure L1 (Redis) hit rate
   - Measure L2 (Bifrost) hit rate
   - Measure L3 (TensorRT) latency
   - Target: 90%+ combined hit rate

5. **Production Deployment**
   - Docker image with TensorRT engine
   - Load balancer across 10-14 GPU instances
   - Monitoring (Langfuse traces)
   - Auto-scaling (K8s HPA)